# FMR â€” Real-Model Pipeline (Kaggle Version)

**Ensure your tokens are set:** Add `HF_TOKEN` (HF read token) and `GH_TOKEN` (GitHub PAT) to the **Kaggle Secrets** menu (Add-ons -> Secrets).\n
**Ensure your tokens are set:** `HF_TOKEN` (HF read token) and `GH_TOKEN` (GitHub PAT, `contents: read+write`).

**Memory-safe by design:** images are downscaled to â‰¤512px (medical scans are high-res and blow up the vision encoder), models load in fp16 and are unloaded between stages, and per-candidate activations are freed. Stages push independently, so a failure never discards earlier results. Start with the **smoke run**.

In [ ]:
# 0. Kaggle Setup (Clone Repo & Configure Git)
import os

# Retrieve tokens from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')
    os.environ['GH_TOKEN'] = user_secrets.get_secret('GH_TOKEN')
    print('Loaded tokens from Kaggle Secrets')
except Exception as e:
    print('Error loading Kaggle secrets. DID YOU CHECK THE BOXES? Please go to Add-ons -> Secrets menu in Kaggle and check the box next to HF_TOKEN and GH_TOKEN to attach them to this notebook.')
    print(e)

if not os.path.exists('fmr-thesis'):
    print('Cloning repository...')
    gh_token = os.environ.get('GH_TOKEN', '')
    clone_cmd = f"git clone https://{gh_token}@github.com/Ankit-blip737/fmr-thesis.git"
    # We hide the token from the output
    res = os.system(clone_cmd)
    if res != 0:
        print("\n!!! GIT CLONE FAILED !!! Please check your GH_TOKEN secret.")

# Move into the repo directory using Kaggle's magic command
get_ipython().run_line_magic('cd', 'fmr-thesis')
print('Changed directory to:', os.getcwd())

# Configure Git for pushing results
!git config --global user.email 'kaggle@fmr.run'
!git config --global user.name 'FMR Kaggle'


In [ ]:
# 1. GPU + memory config (set alloc conf BEFORE importing torch)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce fragmentation (per the OOM hint)
import torch, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:600])
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime -> Change runtime type -> GPU (T4)"
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Install dependencies
import os

if not os.path.exists("fmr/scripts/run_real.py"):
    print("Warning: fmr/scripts/run_real.py not found. Are you running this in the correct directory?")
else:
    print("Found fmr/scripts/run_real.py.")

!python -m pip install -e fmr[real]
!python -m pip install numpy scipy pyyaml matplotlib scikit-learn "transformers>=4.52" datasets accelerate qwen-vl-utils pillow
print("installed")

## 3b. Generate Bounding Boxes for Signal B

We're generating ground-truth bounding boxes for Signal B IoU validation.

In [ ]:
# 3b. Download SLAKE segmentation masks and extract bounding boxes
import os
import zipfile
from huggingface_hub import hf_hub_download

os.makedirs("slake_masks", exist_ok=True)

try:
    masks_path = hf_hub_download(repo_id="BoKelvin/SLAKE", filename="masks.zip", repo_type="dataset", local_dir="slake_masks")
    print("Downloaded SLAKE masks.")
    with zipfile.ZipFile(masks_path, 'r') as zip_ref:
        zip_ref.extractall("slake_masks")
    print("Extracted SLAKE masks.")
except Exception as e:
    print(f"masks not available in mirror or error downloading: {e}; will use SAM fallback")

# Extract bboxes from masks (if available)
if os.path.isdir("slake_masks/masks") or os.path.isdir("slake_masks"):
    !python fmr/scripts/extract_slake_bboxes.py --mask-root slake_masks --image-root slake_imgs --output slake_bboxes.json
else:
    print("SLAKE masks not found; will generate pseudo-labels with SAM")

In [ ]:
# 3c. Generate SAM pseudo-labels for VQA-RAD and PathVQA
!python -m pip install segment-anything-py
# VQA-RAD
!python fmr/scripts/generate_medsam_bboxes.py --dataset vqa_rad --output vqa_rad_bboxes.json --max-samples 5000
# PathVQA
!python fmr/scripts/generate_medsam_bboxes.py --dataset pathvqa --output pathvqa_bboxes.json --max-samples 5000
# SLAKE (if masks extraction failed)
import os
if not os.path.exists("slake_bboxes.json"):
    !python fmr/scripts/generate_medsam_bboxes.py --dataset slake --output slake_bboxes.json --max-samples 5000

## 4. SMOKE RUN â€” do this first (a few minutes, guaranteed headline)

Small VQA-RAD run (80 samples, 3 consistency chains). Produces baselines + the blind-test replication verdict + the conformal gate and **pushes to `master`** â€” so you have a real result before the bigger runs.

In [ ]:
!python fmr/scripts/run_real.py     --dataset vqa_rad --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 80 --n-consistency 3 --alpha 0.20 --push

## 5. Fuller runs â€” scale up once the smoke run succeeded

Each pushes per-stage. If a run still OOMs, add `--max-image-side 448` isn't a CLI flag â€” instead lower `--max-samples`. VQA-RAD and PathVQA have inline images; the SLAKE cell fetches its image archive first.

In [ ]:
# VQA-RAD (X-ray/CT), larger
!python fmr/scripts/run_real.py --dataset vqa_rad --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b --max-samples 5000 --n-consistency 5 --alpha 0.20 --bboxes vqa_rad_bboxes.json --push

In [ ]:
# PathVQA (pathology â€” modality breadth)
!python fmr/scripts/run_real.py --dataset pathvqa --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b --max-samples 5000 --n-consistency 5 --alpha 0.20 --bboxes pathvqa_bboxes.json --push

In [ ]:
# SLAKE â€” fetch images first (mirror is annotations-only), then run with --image-root
import os
import zipfile
from huggingface_hub import hf_hub_download

os.makedirs("slake_imgs", exist_ok=True)

try:
    imgs_path = hf_hub_download(repo_id="BoKelvin/SLAKE", filename="imgs.zip", repo_type="dataset", local_dir="slake_imgs")
    print("Downloaded SLAKE imgs.")
    with zipfile.ZipFile(imgs_path, 'r') as zip_ref:
        zip_ref.extractall("slake_imgs")
    print("Extracted SLAKE imgs.")
except Exception as e:
    print(f"auto-download failed: {e}; place SLAKE imgs under slake_imgs/ manually")

!python fmr/scripts/run_real.py --dataset slake --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b --max-samples 5000 --n-consistency 5 --alpha 0.20 --image-root slake_imgs --bboxes slake_bboxes.json --push

## 6. Inspect what landed (also already pushed to `master`)

In [ ]:
import json, glob
from IPython.display import Image, display
for f in sorted(glob.glob("fmr/outputs/real/*/run_status.json")):
    print(f, "->", json.load(open(f))["status"])
for f in sorted(glob.glob("fmr/outputs/real/*/blind_test.json")):
    rep = json.load(open(f)).get("replication", {})
    print(f"
{f}
  HEADLINE:", rep.get("note"), "| drift slope:", round(rep.get("drift_slope", float('nan')), 4))
for f in sorted(glob.glob("fmr/outputs/real/*/fmr_results.json")):
    r = json.load(open(f)); v = r.get("validation", {}); ab = r["abstention"]
    print(f"
{f}
  per-modality:", r.get("per_modality"))
    print("  AUROC:", {k: round(v[k],3) for k in v if k.startswith("auroc_")})
    for g in ("fs","confidence"):
        print(f"  abstain[{g}] cov={ab[g]['test']['coverage']:.2f} err={ab[g]['test']['retained_error']} AURC={ab[g]['aurc']:.3f}")
for f in sorted(glob.glob("fmr/outputs/real/*/figures/fig1_grounding_drift.png")):
    display(Image(f))